# Linear Probe Pipeline: System vs User Conflict Resolution

## 0. Install

In [2]:
!pip install -q "numpy<2.0" transformer_lens nnterp nnsight scikit-learn pandas plotly tqdm transformers accelerate kaleido

## 1. Config

In [ ]:
from pathlib import Path
import torch
import os

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
DATA_DIR = Path("/content")
HF_TOKEN = ""

BACKEND = "transformerlens"

TOKEN_POSITION = "last_prompt"

EXTRACT_COMPONENTS = False

LABEL_MODE = "binary"
N_CV_FOLDS = 5
N_PERMUTATIONS = 10
TEST_SIZE = 0.2
MAX_SAMPLES = None

SUB_CONSTRAINT = "format"

N_TOP_NEURONS = 50
N_HEATMAP_NEURONS = 200

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print(f"Model: {MODEL_NAME}")
print(f"Backend: {BACKEND}")
print(f"Token position: {TOKEN_POSITION}")
print(f"Extract components (attn_out, mlp_out): {EXTRACT_COMPONENTS}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Model: meta-llama/Llama-3.2-1B-Instruct
Backend: transformerlens
Token position: last_prompt
Extract components (attn_out, mlp_out): False
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


## 2. Imports

In [1]:
import json
import gc
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 3. Data Loading

In [4]:
def load_results(data_dir, model_name):
    safe_name = model_name.replace("/", "_")
    path = Path(data_dir) / f"{safe_name}_results.jsonl"
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


df_all = load_results(DATA_DIR, MODEL_NAME)
df = df_all[df_all["condition"] == "C"].copy()

if LABEL_MODE == "binary":
    df = df[df["label"].isin(["followed_system", "followed_user"])].copy()

df["y"] = (df["label"] == "followed_system").astype(int)

if MAX_SAMPLES is not None:
    df = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=42)

df = df.reset_index(drop=True)
y = df["y"].values

ALL_CONSTRAINT_TYPES = sorted(df["constraint_type"].unique())
ALL_SYS_STRENGTHS = sorted(df["strength"].unique())
ALL_USR_STYLES = sorted(df["user_style"].unique())

print(f"Samples: {len(df)}")
print(f"followed_system: {y.sum()} ({y.mean():.1%})")
print(f"followed_user: {(y == 0).sum()} ({1 - y.mean():.1%})")
print(f"Constraint types: {ALL_CONSTRAINT_TYPES}")
print(f"SCR by constraint type:")
print(df.groupby("constraint_type")["y"].mean())

Samples: 211
followed_system: 14 (6.6%)
followed_user: 197 (93.4%)
Constraint types: ['format', 'language', 'starting_word']
SCR by constraint type:
constraint_type
format           0.122449
language         0.018519
starting_word    0.111111
Name: y, dtype: float64


## 4. Tokenizer & Prompt Utilities

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def build_formatted_prompt(system_text, user_text):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def _encode_len(text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def find_token_positions(system_text, user_text):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    full_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    sys_str = tokenizer.apply_chat_template(
        [{"role": "system", "content": system_text}],
        tokenize=False,
        add_generation_prompt=False,
    )
    sys_user_str = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )

    n_full = _encode_len(full_str)
    n_sys = min(_encode_len(sys_str), n_full - 1)
    n_sys_user = min(_encode_len(sys_user_str), n_full - 1)
    n_sys = max(n_sys, 1)
    n_sys_user = max(n_sys_user, n_sys + 1)

    return {
        "last_prompt": n_full - 1,
        "last_system": n_sys - 1,
        "last_user": n_sys_user - 1,
        "mean_all": (0, n_full),
        "mean_system": (0, n_sys),
        "mean_user": (n_sys, n_sys_user),
        "n_full": n_full,
        "n_sys": n_sys,
        "n_sys_user": n_sys_user,
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

## 5. Precompute Prompts & Token Positions

In [6]:
formatted_prompts = []
position_maps = []
input_ids_list = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Preparing"):
    fp = build_formatted_prompt(row["system_prompt"], row["user_prompt"])
    pm = find_token_positions(row["system_prompt"], row["user_prompt"])
    ids = tokenizer(fp, return_tensors="pt", add_special_tokens=False).input_ids
    formatted_prompts.append(fp)
    position_maps.append(pm)
    input_ids_list.append(ids)

print(f"Prepared {len(formatted_prompts)} prompts")

Preparing:   0%|          | 0/211 [00:00<?, ?it/s]

Prepared 211 prompts


## 6. Activation Extraction

In [7]:
def slice_activation(act_tensor, pos_val):
    if isinstance(pos_val, int):
        return act_tensor[pos_val, :].float().cpu().numpy()
    start, end = pos_val
    return act_tensor[start:end, :].float().mean(dim=0).cpu().numpy()


def build_activation_dict(buffers, n_samples, n_layers, keys):
    return {
        k: np.array(
            [[buffers[k][layer][i] for layer in range(n_layers)] for i in range(n_samples)]
        )
        for k in keys
    }


def save_activations(activations, path):
    np.savez_compressed(path, **activations)
    print(f"Saved: {path}")


def load_activations(path):
    loaded = np.load(path)
    return {k: loaded[k] for k in loaded.files}

In [8]:
pos_key = TOKEN_POSITION
act_keys = ["resid_post"]
if EXTRACT_COMPONENTS:
    act_keys += ["attn_out", "mlp_out"]

if BACKEND == "transformerlens":
    from transformer_lens import HookedTransformer

    model = HookedTransformer.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device=DEVICE,
    )
    model.eval()
    N_LAYERS = model.cfg.n_layers
    D_MODEL = model.cfg.d_model
    print(f"Loaded via TransformerLens | layers={N_LAYERS} d_model={D_MODEL}")

    name_filter_parts = ["resid_post"]
    if EXTRACT_COMPONENTS:
        name_filter_parts += ["attn_out", "mlp_out"]
    names_filter = lambda name: any(name.endswith(p) for p in name_filter_parts)

    buffers = {k: [[] for _ in range(N_LAYERS)] for k in act_keys}

    t0 = time.time()
    for ids, pm in tqdm(zip(input_ids_list, position_maps), total=len(input_ids_list), desc="TL extract"):
        ids_gpu = ids.to(DEVICE)
        with torch.no_grad():
            _, cache = model.run_with_cache(ids_gpu, prepend_bos=False, names_filter=names_filter)
        for layer in range(N_LAYERS):
            resid = cache["resid_post", layer][0]
            buffers["resid_post"][layer].append(slice_activation(resid, pm[pos_key]))
            if EXTRACT_COMPONENTS:
                buffers["attn_out"][layer].append(slice_activation(cache["attn_out", layer][0], pm[pos_key]))
                buffers["mlp_out"][layer].append(slice_activation(cache["mlp_out", layer][0], pm[pos_key]))
        del cache
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    extract_time = time.time() - t0
    print(f"Extraction: {extract_time:.1f}s ({extract_time/len(input_ids_list):.2f}s/sample)")

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

elif BACKEND == "nnterp":
    from nnterp import StandardizedTransformer

    model = StandardizedTransformer(
        MODEL_NAME,
        device_map=DEVICE,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    )
    N_LAYERS = len(list(model.layers))
    D_MODEL = model._model.config.hidden_size
    print(f"Loaded via nnterp | layers={N_LAYERS} d_model={D_MODEL}")

    hf_model = model._model
    buffers = {k: [[] for _ in range(N_LAYERS)] for k in act_keys}

    t0 = time.time()
    for ids, pm in tqdm(zip(input_ids_list, position_maps), total=len(input_ids_list), desc="nnterp extract"):
        ids_gpu = ids.to(DEVICE)
        captured_resid = {}
        captured_attn = {}
        captured_mlp = {}

        def make_resid_hook(layer_idx):
            def hook(module, inp, out):
                hs = out[0].detach()
                if hs.dim() == 3:
                    hs = hs[0]
                captured_resid[layer_idx] = hs.cpu()
            return hook

        def make_attn_hook(layer_idx):
            def hook(module, inp, out):
                hs = out[0].detach()
                if hs.dim() == 3:
                    hs = hs[0]
                captured_attn[layer_idx] = hs.cpu()
            return hook

        def make_mlp_hook(layer_idx):
            def hook(module, inp, out):
                hs = out.detach()
                if hs.dim() == 3:
                    hs = hs[0]
                captured_mlp[layer_idx] = hs.cpu()
            return hook

        handles = []
        for i in range(N_LAYERS):
            handles.append(hf_model.model.layers[i].register_forward_hook(make_resid_hook(i)))
            if EXTRACT_COMPONENTS:
                handles.append(hf_model.model.layers[i].self_attn.register_forward_hook(make_attn_hook(i)))
                handles.append(hf_model.model.layers[i].mlp.register_forward_hook(make_mlp_hook(i)))

        with torch.no_grad():
            hf_model(input_ids=ids_gpu)

        for h in handles:
            h.remove()

        for layer in range(N_LAYERS):
            hs = captured_resid[layer]
            buffers["resid_post"][layer].append(slice_activation(hs, pm[pos_key]))
            if EXTRACT_COMPONENTS:
                buffers["attn_out"][layer].append(slice_activation(captured_attn[layer], pm[pos_key]))
                buffers["mlp_out"][layer].append(slice_activation(captured_mlp[layer], pm[pos_key]))

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    extract_time = time.time() - t0
    print(f"Extraction: {extract_time:.1f}s ({extract_time/len(input_ids_list):.2f}s/sample)")

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

else:
    raise ValueError(f"Unknown backend: {BACKEND}")

activations = build_activation_dict(buffers, len(input_ids_list), N_LAYERS, act_keys)
del buffers

safe_model = MODEL_NAME.replace('/', '_')
save_activations(activations, f"act_{BACKEND}_{safe_model}.npz")

for k, v in activations.items():
    print(f"  {k}: {v.shape}")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer
Loaded via TransformerLens | layers=16 d_model=2048


TL extract:   0%|          | 0/211 [00:00<?, ?it/s]

Extraction: 9.9s (0.05s/sample)
Saved: act_transformerlens_meta-llama_Llama-3.2-1B-Instruct.npz
  resid_post: (211, 16, 2048)


## 7. Linear Probe (per layer)

In [9]:
def probe_per_layer(X, y, n_folds=N_CV_FOLDS):
    n_layers = X.shape[1]
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    rows = []
    for layer in tqdm(range(n_layers), desc="Probing"):
        X_layer = X[:, layer, :]
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")),
        ])
        scores = cross_val_score(pipe, X_layer, y, cv=cv, scoring="balanced_accuracy")
        rows.append({"layer": layer, "mean": scores.mean(), "std": scores.std()})
    return pd.DataFrame(rows)


probe_results = probe_per_layer(activations["resid_post"], y)
best_probe = probe_results.loc[probe_results["mean"].idxmax()]
PEAK_LAYER = int(best_probe["layer"])

print(f"Peak: {best_probe['mean']:.3f} +/- {best_probe['std']:.3f} @ layer {PEAK_LAYER}")

Probing:   0%|          | 0/16 [00:00<?, ?it/s]

Peak: 0.933 +/- 0.133 @ layer 9


## 8. Permuted-Label Control

In [10]:
rng = np.random.default_rng(0)

def probe_permuted(X, y, n_permutations=N_PERMUTATIONS, n_folds=N_CV_FOLDS):
    n_layers = X.shape[1]
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    layer_scores = np.zeros((n_permutations, n_layers))
    for p in tqdm(range(n_permutations), desc="Permuted control"):
        y_perm = rng.permutation(y)
        for layer in range(n_layers):
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")),
            ])
            scores = cross_val_score(pipe, X[:, layer, :], y_perm, cv=cv, scoring="balanced_accuracy")
            layer_scores[p, layer] = scores.mean()
    rows = [{"layer": l, "mean": layer_scores[:, l].mean(), "std": layer_scores[:, l].std()} for l in range(n_layers)]
    return pd.DataFrame(rows)


perm_results = probe_permuted(activations["resid_post"], y)
best_perm = perm_results.loc[perm_results["mean"].idxmax()]

print(f"Permuted control peak: {best_perm['mean']:.3f} (should be ~0.50)")
print(f"Real probe peak: {best_probe['mean']:.3f}")
print(f"Gap: +{best_probe['mean'] - best_perm['mean']:.3f}")

Permuted control:   0%|          | 0/10 [00:00<?, ?it/s]

Permuted control peak: 0.514 (should be ~0.50)
Real probe peak: 0.933
Gap: +0.419


## 9. Metadata-Only Control

In [11]:
def one_hot(val, categories):
    return [int(val == c) for c in categories]


def build_metadata_features(df_sub, position_maps_sub):
    return np.array([
        [
            pm["last_prompt"] + 1,
            pm["mean_system"][1],
            pm["mean_user"][1] - pm["mean_user"][0],
            pm["mean_user"][0],
            *one_hot(row["constraint_type"], ALL_CONSTRAINT_TYPES),
            *one_hot(row["strength"], ALL_SYS_STRENGTHS),
            *one_hot(row["user_style"], ALL_USR_STYLES),
            int(row["direction"] == "b_to_a"),
        ]
        for pm, (_, row) in zip(position_maps_sub, df_sub.iterrows())
    ], dtype=np.float32)


X_meta = build_metadata_features(df, position_maps)

cv = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
pipe_meta = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")),
])
meta_scores = cross_val_score(pipe_meta, X_meta, y, cv=cv, scoring="balanced_accuracy")
meta_acc = meta_scores.mean()
meta_std = meta_scores.std()

print(f"Metadata-only control: {meta_acc:.3f} +/- {meta_std:.3f}")
print(f"Probe peak: {best_probe['mean']:.3f}")
print(f"Gap: +{best_probe['mean'] - meta_acc:.3f}")

Metadata-only control: 0.631 +/- 0.124
Probe peak: 0.933
Gap: +0.303


## 10. Train/Test Split & Probe Direction Extraction

In [28]:
X_resid = activations["resid_post"]
n_layers = X_resid.shape[1]

X_train_idx, X_test_idx = train_test_split(
    np.arange(len(y)), test_size=TEST_SIZE, random_state=42, stratify=y
)
y_train, y_test = y[X_train_idx], y[X_test_idx]

print(f"Train: {len(y_train)} samples ({y_train.sum()} pos, {(y_train==0).sum()} neg)")
print(f"Test:  {len(y_test)} samples ({y_test.sum()} pos, {(y_test==0).sum()} neg)")

if y_test.sum() < 3:
    print(f"WARNING: only {y_test.sum()} positive samples in test set. Test accuracy will be noisy.")

train_accs = []
test_accs = []
probe_directions = []

for layer in tqdm(range(n_layers), desc="Train/test probing"):
    X_layer = X_resid[:, layer, :]
    X_tr, X_te = X_layer[X_train_idx], X_layer[X_test_idx]

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    clf = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")
    clf.fit(X_tr_s, y_train)

    train_accs.append(balanced_accuracy_score(y_train, clf.predict(X_tr_s)))
    test_accs.append(balanced_accuracy_score(y_test, clf.predict(X_te_s)))
    probe_directions.append(clf.coef_[0].copy())

df_train_test = pd.DataFrame({
    "layer": range(n_layers),
    "train_acc": train_accs,
    "test_acc": test_accs,
})

PEAK_LAYER_TT = PEAK_LAYER
precedence_direction = probe_directions[PEAK_LAYER_TT]

print(f"\nUsing CV-determined peak layer: {PEAK_LAYER_TT}")
print(f"  Train acc @ peak: {train_accs[PEAK_LAYER_TT]:.3f}")
print(f"  Test acc @ peak:  {test_accs[PEAK_LAYER_TT]:.3f}")
print(f"  Overfit gap:      {train_accs[PEAK_LAYER_TT] - test_accs[PEAK_LAYER_TT]:.3f}")

best_test_layer = int(df_train_test["test_acc"].idxmax())
print(f"\nBest test-acc layer: {best_test_layer} ({test_accs[best_test_layer]:.3f})")
if best_test_layer != PEAK_LAYER_TT:
    print(f"  (differs from CV peak; with {y_test.sum()} positive test samples, test acc is high-variance)")

print(f"\nPrecedence direction shape: {precedence_direction.shape}")
print(f"Precedence direction norm: {np.linalg.norm(precedence_direction):.4f}")

Train: 168 samples (11 pos, 157 neg)
Test:  43 samples (3 pos, 40 neg)


Train/test probing:   0%|          | 0/16 [00:00<?, ?it/s]


Using CV-determined peak layer: 9
  Train acc @ peak: 1.000
  Test acc @ peak:  0.642
  Overfit gap:      0.358

Best test-acc layer: 0 (0.667)
  (differs from CV peak; with 3 positive test samples, test acc is high-variance)

Precedence direction shape: (2048,)
Precedence direction norm: 1.1223


## 11. Statistical Significance

In [31]:
N_BOOTSTRAP = 1000
rng_boot = np.random.default_rng(42)

X_peak_all = X_resid[:, PEAK_LAYER, :]

cv_sig = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)

probe_fold_scores = []
meta_fold_scores = []

for train_idx, val_idx in cv_sig.split(X_peak_all, y):
    pipe_p = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs"))])
    pipe_p.fit(X_peak_all[train_idx], y[train_idx])
    probe_fold_scores.append(balanced_accuracy_score(y[val_idx], pipe_p.predict(X_peak_all[val_idx])))

    pipe_m = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs"))])
    pipe_m.fit(X_meta[train_idx], y[train_idx])
    meta_fold_scores.append(balanced_accuracy_score(y[val_idx], pipe_m.predict(X_meta[val_idx])))

probe_fold_scores = np.array(probe_fold_scores)
meta_fold_scores = np.array(meta_fold_scores)
observed_gap = (probe_fold_scores - meta_fold_scores).mean()

boot_gaps = []
for _ in range(N_BOOTSTRAP):
    idx = rng_boot.choice(N_CV_FOLDS, size=N_CV_FOLDS, replace=True)
    boot_gaps.append((probe_fold_scores[idx] - meta_fold_scores[idx]).mean())
boot_gaps = np.array(boot_gaps)

ci_lower = np.percentile(boot_gaps, 2.5)
ci_upper = np.percentile(boot_gaps, 97.5)
p_value = (boot_gaps <= 0).mean()

print(f"Observed gap (probe - metadata): {observed_gap:.3f}")
print(f"95% CI: [{ci_lower:.3f}, {ci_upper:.3f}]")
print(f"Bootstrap p-value (gap <= 0): {p_value:.4f}")
print(f"Significant at 0.05: {p_value < 0.05}")

N_PERM_TEST = 100
perm_p_rng = np.random.default_rng(123)

real_score = cross_val_score(
    Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs"))]),
    X_peak_all, y, cv=cv_sig, scoring="balanced_accuracy"
).mean()

perm_scores = []
for _ in tqdm(range(N_PERM_TEST), desc="Permutation test"):
    y_shuf = perm_p_rng.permutation(y)
    s = cross_val_score(
        Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs"))]),
        X_peak_all, y_shuf, cv=cv_sig, scoring="balanced_accuracy"
    ).mean()
    perm_scores.append(s)
perm_scores = np.array(perm_scores)

perm_p_value = (perm_scores >= real_score).mean()
print(f"\nPermutation test ({N_PERM_TEST} permutations):")
print(f"Real score: {real_score:.3f}")
print(f"Permuted mean: {perm_scores.mean():.3f} +/- {perm_scores.std():.3f}")
print(f"p-value: {perm_p_value:.4f}")

Observed gap (probe - metadata): 0.303
95% CI: [0.141, 0.467]
Bootstrap p-value (gap <= 0): 0.0000
Significant at 0.05: True


Permutation test:   0%|          | 0/100 [00:00<?, ?it/s]


Permutation test (100 permutations):
Real score: 0.933
Permuted mean: 0.499 +/- 0.038
p-value: 0.0000


## 12. Accuracy by Layer Plot (Train & Test)

In [14]:
fig = go.Figure()

layers = df_train_test["layer"].values

fig.add_trace(go.Scatter(
    x=layers, y=df_train_test["train_acc"].values,
    mode="lines", name="Train",
    line=dict(color="#ff7f0e", width=2),
))
fig.add_trace(go.Scatter(
    x=layers, y=df_train_test["test_acc"].values,
    mode="lines", name="Test",
    line=dict(color="#1f77b4", width=2),
))

cv_means = probe_results["mean"].values
cv_stds = probe_results["std"].values
fig.add_trace(go.Scatter(
    x=layers, y=cv_means, mode="lines", name=f"CV ({N_CV_FOLDS}-fold)",
    line=dict(color="#2ca02c", width=2, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=np.concatenate([layers, layers[::-1]]),
    y=np.concatenate([cv_means + cv_stds, (cv_means - cv_stds)[::-1]]),
    fill="toself", fillcolor="rgba(44,160,44,0.15)",
    line=dict(color="rgba(0,0,0,0)"), showlegend=False,
))

fig.add_hline(y=meta_acc, line_dash="dot", line_color="red",
              annotation_text=f"Metadata ctrl: {meta_acc:.3f}")
fig.add_hline(y=0.5, line_dash="dot", line_color="gray",
              annotation_text="Chance")

fig.update_layout(
    title=f"Probe Accuracy by Layer<br><sub>{TOKEN_POSITION} | {MODEL_NAME} | {BACKEND}</sub>",
    xaxis_title="Layer", yaxis_title="Balanced Accuracy",
    yaxis_range=[0.3, 1.05],
    width=900, height=500, template="plotly_white",
)
fig.show()

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




## 13. Raw Activation Heatmap

In [15]:
X_resid_all = activations["resid_post"]
n_samples_h, n_layers_h, d_model_h = X_resid_all.shape

X_mean = X_resid_all.mean(axis=0)

neuron_step = max(1, d_model_h // N_HEATMAP_NEURONS)
neuron_indices = np.arange(0, d_model_h, neuron_step)[:N_HEATMAP_NEURONS]
heatmap_data = X_mean[:, neuron_indices].T

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data,
    x=[f"L{i}" for i in range(n_layers_h)],
    y=[f"N{i}" for i in neuron_indices],
    colorscale="Viridis",
    colorbar=dict(title="Activation"),
))
fig.update_layout(
    title=f"Raw Activation Values (Mean Across Samples)<br><sub>{TOKEN_POSITION} | {N_HEATMAP_NEURONS} neurons sampled from {d_model_h}</sub>",
    xaxis_title="Layer", yaxis_title="Neuron Index",
    width=1200, height=800, template="plotly_white",
)
fig.show()

## 14. Neuron Importance Analysis

In [16]:
all_weights = np.zeros((n_layers, D_MODEL))

for layer in tqdm(range(n_layers), desc="Neuron importance"):
    X_layer = X_resid[:, layer, :]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_layer)
    clf = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")
    clf.fit(X_scaled, y)
    all_weights[layer, :] = clf.coef_[0]

global_importance = np.abs(all_weights).max(axis=0)
top_neurons = np.argsort(global_importance)[::-1][:N_TOP_NEURONS]

print(f"Top {N_TOP_NEURONS} neurons by global importance:")
for rank, n_idx in enumerate(top_neurons[:10]):
    print(f"  #{rank+1} N{n_idx}: max|w|={global_importance[n_idx]:.4f} (peak at layer {np.abs(all_weights[:, n_idx]).argmax()})")

Neuron importance:   0%|          | 0/16 [00:00<?, ?it/s]

Top 50 neurons by global importance:
  #1 N154: max|w|=0.2618 (peak at layer 0)
  #2 N771: max|w|=0.2616 (peak at layer 0)
  #3 N1403: max|w|=0.2498 (peak at layer 0)
  #4 N768: max|w|=0.2352 (peak at layer 2)
  #5 N452: max|w|=0.2289 (peak at layer 0)
  #6 N1181: max|w|=0.2284 (peak at layer 2)
  #7 N1191: max|w|=0.2223 (peak at layer 0)
  #8 N795: max|w|=0.2080 (peak at layer 0)
  #9 N1670: max|w|=0.2067 (peak at layer 2)
  #10 N399: max|w|=0.2045 (peak at layer 1)


## 15. Activation Difference Heatmap

In [17]:
system_mask = (y == 1)
user_mask = (y == 0)

activation_diff = np.zeros((N_TOP_NEURONS, n_layers))
for i, neuron_idx in enumerate(top_neurons):
    for layer in range(n_layers):
        acts = X_resid[:, layer, neuron_idx]
        activation_diff[i, layer] = acts[system_mask].mean() - acts[user_mask].mean()

fig = go.Figure(data=go.Heatmap(
    z=activation_diff,
    x=[f"L{i}" for i in range(n_layers)],
    y=[f"N{i}" for i in top_neurons],
    colorscale="RdBu_r",
    zmid=0,
    colorbar=dict(title="Mean Diff<br>(sys - usr)"),
))
fig.update_layout(
    title=f"Activation Difference (System - User) | Top {N_TOP_NEURONS} Neurons<br><sub>{TOKEN_POSITION} | {MODEL_NAME}</sub>",
    xaxis_title="Layer", yaxis_title="Neuron",
    width=1200, height=800, template="plotly_white",
)
fig.show()

print(f"Top 5 neurons by |diff| at peak layer {PEAK_LAYER}:")
peak_diffs = np.abs(activation_diff[:, PEAK_LAYER])
top_at_peak = np.argsort(peak_diffs)[::-1][:5]
for rank, idx in enumerate(top_at_peak):
    n_idx = top_neurons[idx]
    diff = activation_diff[idx, PEAK_LAYER]
    print(f"  #{rank+1} N{n_idx}: diff={diff:.4f} ({'system higher' if diff > 0 else 'user higher'})")

Top 5 neurons by |diff| at peak layer 9:
  #1 N1349: diff=-0.1201 (user higher)
  #2 N662: diff=-0.0579 (user higher)
  #3 N1022: diff=-0.0571 (user higher)
  #4 N39: diff=-0.0531 (user higher)
  #5 N1572: diff=0.0514 (system higher)


## 16. PCA Analysis

In [18]:
from sklearn.decomposition import PCA

X_peak = X_resid[:, PEAK_LAYER, :]
scaler_pca = StandardScaler()
X_peak_scaled = scaler_pca.fit_transform(X_peak)

pca = PCA(n_components=min(50, X_peak_scaled.shape[0], X_peak_scaled.shape[1]))
X_pca = pca.fit_transform(X_peak_scaled)

fig = make_subplots(rows=1, cols=2, subplot_titles=["PC1 vs PC2", "Explained Variance Ratio"])

for label_val, label_name, color in [(1, "followed_system", "#d62728"), (0, "followed_user", "#1f77b4")]:
    mask = y == label_val
    fig.add_trace(go.Scatter(
        x=X_pca[mask, 0], y=X_pca[mask, 1],
        mode="markers", name=label_name,
        marker=dict(color=color, size=8, opacity=0.7),
    ), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(range(1, len(pca.explained_variance_ratio_) + 1)),
    y=np.cumsum(pca.explained_variance_ratio_),
    mode="lines+markers", name="Cumulative",
    line=dict(color="#2ca02c", width=2),
    marker=dict(size=4),
), row=1, col=2)

fig.add_trace(go.Bar(
    x=list(range(1, len(pca.explained_variance_ratio_) + 1)),
    y=pca.explained_variance_ratio_,
    name="Per component",
    marker_color="#1f77b4", opacity=0.5,
), row=1, col=2)

fig.update_xaxes(title_text="PC1", row=1, col=1)
fig.update_yaxes(title_text="PC2", row=1, col=1)
fig.update_xaxes(title_text="Component", row=1, col=2)
fig.update_yaxes(title_text="Variance Ratio", row=1, col=2)

fig.update_layout(
    title=f"PCA of Activations @ Layer {PEAK_LAYER}<br><sub>{MODEL_NAME} | {TOKEN_POSITION}</sub>",
    width=1200, height=500, template="plotly_white",
)
fig.show()

prec_dir_in_pca = pca.components_ @ precedence_direction
prec_dir_in_pca_norm = prec_dir_in_pca / np.linalg.norm(prec_dir_in_pca)

cos_with_pcs = np.abs(prec_dir_in_pca_norm)

print(f"Explained variance (PC1): {pca.explained_variance_ratio_[0]:.3f}")
print(f"Explained variance (PC1+PC2): {sum(pca.explained_variance_ratio_[:2]):.3f}")
print(f"Explained variance (top 10): {sum(pca.explained_variance_ratio_[:10]):.3f}")
print(f"\n|cos| of precedence direction with top PCs:")
for i in range(min(10, len(cos_with_pcs))):
    print(f"  PC{i+1}: |cos|={cos_with_pcs[i]:.4f}")

Explained variance (PC1): 0.246
Explained variance (PC1+PC2): 0.390
Explained variance (top 10): 0.801

|cos| of precedence direction with top PCs:
  PC1: |cos|=0.0382
  PC2: |cos|=0.3600
  PC3: |cos|=0.1223
  PC4: |cos|=0.0075
  PC5: |cos|=0.0710
  PC6: |cos|=0.2266
  PC7: |cos|=0.0901
  PC8: |cos|=0.0206
  PC9: |cos|=0.0266
  PC10: |cos|=0.1308


## 17. Projection Gap Analysis

In [19]:
prec_dir_unit = precedence_direction / np.linalg.norm(precedence_direction)

projections_by_layer = []
gaps_by_layer = []

for layer in range(n_layers):
    X_layer = X_resid[:, layer, :]
    scaler_tmp = StandardScaler()
    X_scaled_tmp = scaler_tmp.fit_transform(X_layer)

    clf_tmp = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")
    clf_tmp.fit(X_scaled_tmp, y)
    direction = clf_tmp.coef_[0]
    direction_unit = direction / np.linalg.norm(direction)

    proj = X_scaled_tmp @ direction_unit
    projections_by_layer.append(proj)

    sys_mean = proj[y == 1].mean() if (y == 1).any() else 0
    usr_mean = proj[y == 0].mean() if (y == 0).any() else 0
    sys_std = proj[y == 1].std() if (y == 1).sum() > 1 else 1
    usr_std = proj[y == 0].std() if (y == 0).sum() > 1 else 1
    pooled_std = np.sqrt((sys_std**2 + usr_std**2) / 2)
    gap = (sys_mean - usr_mean) / pooled_std if pooled_std > 0 else 0
    gaps_by_layer.append(gap)

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    f"Projection Histogram @ Layer {PEAK_LAYER}",
    "Projection Gap (Cohen's d) by Layer"
])

proj_peak = projections_by_layer[PEAK_LAYER]
for label_val, label_name, color in [(1, "followed_system", "#d62728"), (0, "followed_user", "#1f77b4")]:
    mask = y == label_val
    fig.add_trace(go.Histogram(
        x=proj_peak[mask], name=label_name,
        marker_color=color, opacity=0.6,
        nbinsx=30,
    ), row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(range(n_layers)), y=gaps_by_layer,
    mode="lines+markers", name="Gap",
    line=dict(color="#1f77b4", width=2),
    marker=dict(size=4),
    showlegend=False,
), row=1, col=2)

fig.update_xaxes(title_text="Projection onto precedence direction", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_xaxes(title_text="Layer", row=1, col=2)
fig.update_yaxes(title_text="Cohen's d", row=1, col=2)

fig.update_layout(
    title=f"Projection Gap Analysis<br><sub>{MODEL_NAME} | {TOKEN_POSITION}</sub>",
    width=1200, height=500, template="plotly_white",
    barmode="overlay",
)
fig.show()

print(f"Projection gap at peak layer {PEAK_LAYER}: Cohen's d = {gaps_by_layer[PEAK_LAYER]:.3f}")
print(f"Max gap: d = {max(gaps_by_layer):.3f} @ layer {np.argmax(gaps_by_layer)}")

Projection gap at peak layer 9: Cohen's d = 4.846
Max gap: d = 6.980 @ layer 15


## 18. Decision Boundary Visualization

In [20]:
X_2d = X_pca[:, :2]

clf_2d = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")
clf_2d.fit(X_2d, y)
acc_2d = balanced_accuracy_score(y, clf_2d.predict(X_2d))

x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
Z = clf_2d.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

fig = go.Figure()

fig.add_trace(go.Contour(
    x=np.linspace(x_min, x_max, 200),
    y=np.linspace(y_min, y_max, 200),
    z=Z,
    colorscale=[[0, "#1f77b4"], [0.5, "#f0f0f0"], [1, "#d62728"]],
    opacity=0.4,
    showscale=True,
    colorbar=dict(title="P(system)"),
    contours=dict(showlines=True),
))

fig.add_trace(go.Contour(
    x=np.linspace(x_min, x_max, 200),
    y=np.linspace(y_min, y_max, 200),
    z=Z,
    showscale=False,
    contours=dict(
        type="constraint",
        operation="=",
        value=0.5,
    ),
    line=dict(color="black", width=3),
    name="Decision boundary",
))

for label_val, label_name, color in [(1, "followed_system", "#d62728"), (0, "followed_user", "#1f77b4")]:
    mask = y == label_val
    fig.add_trace(go.Scatter(
        x=X_2d[mask, 0], y=X_2d[mask, 1],
        mode="markers", name=label_name,
        marker=dict(color=color, size=8, opacity=0.8, line=dict(width=1, color="white")),
    ))

w = clf_2d.coef_[0]
w_norm = w / np.linalg.norm(w)
center = X_2d.mean(axis=0)
arrow_len = (x_max - x_min) * 0.15
fig.add_annotation(
    x=center[0] + w_norm[0] * arrow_len,
    y=center[1] + w_norm[1] * arrow_len,
    ax=center[0], ay=center[1],
    xref="x", yref="y", axref="x", ayref="y",
    showarrow=True, arrowhead=3, arrowsize=1.5, arrowwidth=2, arrowcolor="black",
)

fig.update_layout(
    title=f"Decision Boundary in PCA Space @ Layer {PEAK_LAYER}<br><sub>2D probe accuracy: {acc_2d:.3f} | Variance explained: {sum(pca.explained_variance_ratio_[:2]):.1%}</sub>",
    xaxis_title="PC1", yaxis_title="PC2",
    width=800, height=700, template="plotly_white",
)
fig.show()

print(f"2D probe accuracy (PC1+PC2 only): {acc_2d:.3f}")
print(f"Full probe accuracy (all {D_MODEL} dims): {best_probe['mean']:.3f}")
print(f"Variance in PC1+PC2: {sum(pca.explained_variance_ratio_[:2]):.1%}")
print(f"Decision boundary normal direction angle from PC1: {np.degrees(np.arctan2(w_norm[1], w_norm[0])):.1f} deg")

2D probe accuracy (PC1+PC2 only): 0.500
Full probe accuracy (all 2048 dims): 0.933
Variance in PC1+PC2: 39.0%
Decision boundary normal direction angle from PC1: -128.2 deg


## 19. Constraint-Specific Sub-Analysis

In [21]:
df_sub = df[df["constraint_type"] == SUB_CONSTRAINT].copy().reset_index(drop=True)
y_sub = df_sub["y"].values
sub_idx = df[df["constraint_type"] == SUB_CONSTRAINT].index.values

print(f"Constraint: {SUB_CONSTRAINT}")
print(f"Samples: {len(df_sub)}")
print(f"SCR: {y_sub.mean():.1%}")

act_sub = activations["resid_post"][sub_idx]
pm_sub = [position_maps[i] for i in sub_idx]

n_folds_sub = min(N_CV_FOLDS, max(2, min(y_sub.sum(), (y_sub == 0).sum())))
cv_sub = StratifiedKFold(n_splits=n_folds_sub, shuffle=True, random_state=42)

rows_sub = []
for layer in range(n_layers):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=500, C=1.0, solver="lbfgs")),
    ])
    scores = cross_val_score(pipe, act_sub[:, layer, :], y_sub, cv=cv_sub, scoring="balanced_accuracy")
    rows_sub.append({"layer": layer, "mean": scores.mean(), "std": scores.std()})
probe_sub = pd.DataFrame(rows_sub)
best_sub = probe_sub.loc[probe_sub["mean"].idxmax()]

X_meta_sub = build_metadata_features(df_sub, pm_sub)
pipe_meta_sub = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")),
])
meta_sub_scores = cross_val_score(pipe_meta_sub, X_meta_sub, y_sub, cv=cv_sub, scoring="balanced_accuracy")
meta_sub_acc = meta_sub_scores.mean()

perm_sub_accs = []
X_sub_peak = act_sub[:, int(best_sub["layer"]), :]
for _ in range(N_PERMUTATIONS):
    y_shuf = rng.permutation(y_sub)
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=500, C=1.0, solver="lbfgs"))])
    scores = cross_val_score(pipe, X_sub_peak, y_shuf, cv=cv_sub, scoring="balanced_accuracy")
    perm_sub_accs.append(scores.mean())
perm_sub_mean = np.mean(perm_sub_accs)

print(f"\nProbe peak: {best_sub['mean']:.3f} +/- {best_sub['std']:.3f} @ layer {int(best_sub['layer'])}")
print(f"Metadata control: {meta_sub_acc:.3f}")
print(f"Permuted control: {perm_sub_mean:.3f}")
print(f"Gap (probe - metadata): {best_sub['mean'] - meta_sub_acc:+.3f}")
print(f"Gap (probe - permuted): {best_sub['mean'] - perm_sub_mean:+.3f}")

Constraint: format
Samples: 49
SCR: 12.2%

Probe peak: 1.000 +/- 0.000 @ layer 0
Metadata control: 0.489
Permuted control: 0.493
Gap (probe - metadata): +0.511
Gap (probe - permuted): +0.507


## 20. Leave-One-Constraint-Type-Out Generalization

In [22]:
print("Leave-one-constraint-type-out generalization:\n")

loco_results = []

for held_out in ALL_CONSTRAINT_TYPES:
    train_mask = df["constraint_type"] != held_out
    test_mask = df["constraint_type"] == held_out

    y_tr = y[train_mask]
    y_te = y[test_mask]

    if y_te.sum() == 0 or (y_te == 0).sum() == 0:
        print(f"  {held_out}: skipped (no class variation in test set)")
        loco_results.append({"held_out": held_out, "layer": -1, "train_acc": np.nan, "test_acc": np.nan})
        continue

    best_test = -1
    best_layer = -1
    best_train = -1

    for layer in range(n_layers):
        X_tr = X_resid[train_mask, layer, :]
        X_te = X_resid[test_mask, layer, :]

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        clf = LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs")
        clf.fit(X_tr_s, y_tr)

        tr_acc = balanced_accuracy_score(y_tr, clf.predict(X_tr_s))
        te_acc = balanced_accuracy_score(y_te, clf.predict(X_te_s))

        if te_acc > best_test:
            best_test = te_acc
            best_layer = layer
            best_train = tr_acc

    loco_results.append({"held_out": held_out, "layer": best_layer, "train_acc": best_train, "test_acc": best_test})
    print(f"  Hold out '{held_out}': test_acc={best_test:.3f} (train={best_train:.3f}) @ layer {best_layer}")

df_loco = pd.DataFrame(loco_results)
print(f"\nMean cross-constraint test acc: {df_loco['test_acc'].dropna().mean():.3f}")

Leave-one-constraint-type-out generalization:

  Hold out 'format': test_acc=0.870 (train=1.000) @ layer 6
  Hold out 'language': test_acc=0.684 (train=1.000) @ layer 5
  Hold out 'starting_word': test_acc=0.500 (train=1.000) @ layer 0

Mean cross-constraint test acc: 0.685


## 21. Per-Token Decision Transition Curve

In [23]:
target_layer = PEAK_LAYER

print(f"Re-loading model for per-token extraction at layer {target_layer}...")

all_token_acts = []

if BACKEND == "transformerlens":
    from transformer_lens import HookedTransformer
    _model = HookedTransformer.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device=DEVICE,
    )
    _model.eval()
    for ids in tqdm(input_ids_list, desc="Per-token extract"):
        ids_gpu = ids.to(DEVICE)
        with torch.no_grad():
            _, cache = _model.run_with_cache(
                ids_gpu, prepend_bos=False,
                names_filter=lambda name: name.endswith("resid_post"),
            )
        all_token_acts.append(cache["resid_post", target_layer][0].float().cpu().numpy())
        del cache
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    del _model

elif BACKEND == "nnterp":
    from nnterp import StandardizedTransformer
    _model = StandardizedTransformer(
        MODEL_NAME,
        device_map=DEVICE,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    )
    _hf = _model._model
    for ids in tqdm(input_ids_list, desc="Per-token extract"):
        ids_gpu = ids.to(DEVICE)
        captured = {}
        def _hook(module, inp, out):
            hs = out[0].detach()
            if hs.dim() == 3:
                hs = hs[0]
            captured[0] = hs.cpu()
        h = _hf.model.layers[target_layer].register_forward_hook(_hook)
        with torch.no_grad():
            _hf(input_ids=ids_gpu)
        h.remove()
        all_token_acts.append(captured[0].float().numpy())
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    del _model

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

min_seq_len = min(pm["n_full"] for pm in position_maps)

token_accs = []
token_counts = []

for tok_pos in tqdm(range(min_seq_len), desc="Per-token probing"):
    vecs = []
    labels = []
    for i in range(len(y)):
        if tok_pos < all_token_acts[i].shape[0]:
            vecs.append(all_token_acts[i][tok_pos, :])
            labels.append(y[i])

    if len(set(labels)) < 2 or len(labels) < 10:
        token_accs.append(np.nan)
        token_counts.append(len(labels))
        continue

    X_tok = np.array(vecs)
    y_tok = np.array(labels)

    n_folds_tok = min(N_CV_FOLDS, min(y_tok.sum(), (y_tok == 0).sum()))
    if n_folds_tok < 2:
        token_accs.append(np.nan)
        token_counts.append(len(labels))
        continue

    cv_tok = StratifiedKFold(n_splits=n_folds_tok, shuffle=True, random_state=42)
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=500, C=1.0, solver="lbfgs"))])
    scores = cross_val_score(pipe, X_tok, y_tok, cv=cv_tok, scoring="balanced_accuracy")
    token_accs.append(scores.mean())
    token_counts.append(len(labels))

del all_token_acts
gc.collect()

token_positions_arr = np.arange(min_seq_len)
token_accs_arr = np.array(token_accs)

median_sys_end = int(np.median([pm["n_sys"] for pm in position_maps]))
median_user_end = int(np.median([pm["n_sys_user"] for pm in position_maps]))

print(f"Peak token accuracy: {np.nanmax(token_accs_arr):.3f} @ token {np.nanargmax(token_accs_arr)}")

Re-loading model for per-token extraction at layer 9...


Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer


Per-token extract:   0%|          | 0/211 [00:00<?, ?it/s]

Per-token probing:   0%|          | 0/59 [00:00<?, ?it/s]

Peak token accuracy: 0.609 @ token 39


In [24]:
fig = go.Figure()

valid = ~np.isnan(token_accs_arr)
fig.add_trace(go.Scatter(
    x=token_positions_arr[valid], y=token_accs_arr[valid],
    mode="lines+markers", name="Probe accuracy",
    line=dict(color="#1f77b4", width=2),
    marker=dict(size=3),
))

fig.add_hline(y=0.5, line_dash="dot", line_color="gray")
fig.add_vline(x=median_sys_end, line_dash="dash", line_color="orange",
              annotation_text="system end")
fig.add_vline(x=median_user_end, line_dash="dash", line_color="red",
              annotation_text="user end")

fig.update_layout(
    title=f"Per-Token Decision Transition @ Layer {target_layer}<br><sub>{MODEL_NAME} | Probe accuracy at each token position</sub>",
    xaxis_title="Token Position", yaxis_title="Balanced Accuracy",
    yaxis_range=[0.3, 1.05],
    width=1000, height=500, template="plotly_white",
)
fig.show()

print(f"Median system end: token {median_sys_end}")
print(f"Median user end: token {median_user_end}")
print(f"Peak token accuracy: {np.nanmax(token_accs_arr):.3f} @ token {np.nanargmax(token_accs_arr)}")

Median system end: token 51
Median user end: token 76
Peak token accuracy: 0.609 @ token 39


## 22. Probing Different Internal Representations

In [25]:
if not EXTRACT_COMPONENTS:
    print("Skipped: set EXTRACT_COMPONENTS = True in config to enable.")
else:
    component_results = {}
    for comp_name in ["resid_post", "attn_out", "mlp_out"]:
        print(f"Probing {comp_name}...")
        comp_results = probe_per_layer(activations[comp_name], y)
        component_results[comp_name] = comp_results
        best_c = comp_results.loc[comp_results["mean"].idxmax()]
        print(f"  Peak: {best_c['mean']:.3f} +/- {best_c['std']:.3f} @ layer {int(best_c['layer'])}")

    fig = go.Figure()
    colors = {"resid_post": "#1f77b4", "attn_out": "#ff7f0e", "mlp_out": "#2ca02c"}
    for comp_name, df_c in component_results.items():
        fig.add_trace(go.Scatter(
            x=df_c["layer"].values, y=df_c["mean"].values,
            mode="lines", name=comp_name,
            line=dict(color=colors[comp_name], width=2),
        ))
    fig.add_hline(y=0.5, line_dash="dot", line_color="gray")
    fig.add_hline(y=meta_acc, line_dash="dot", line_color="red",
                  annotation_text=f"Metadata ctrl: {meta_acc:.3f}")
    fig.update_layout(
        title=f"Probe Accuracy by Component<br><sub>{TOKEN_POSITION} | {MODEL_NAME}</sub>",
        xaxis_title="Layer", yaxis_title="Balanced Accuracy",
        yaxis_range=[0.3, 1.05],
        width=900, height=500, template="plotly_white",
    )
    fig.show()

Skipped: set EXTRACT_COMPONENTS = True in config to enable.


## 23. Summary

In [32]:
print("=" * 60)
print(f"MODEL: {MODEL_NAME}")
print(f"BACKEND: {BACKEND}")
print(f"TOKEN POSITION: {TOKEN_POSITION}")
print(f"SAMPLES: {len(df)} (SCR: {y.mean():.1%})")
print("=" * 60)

print(f"\nProbe (CV) peak: {best_probe['mean']:.3f} +/- {best_probe['std']:.3f} @ layer {PEAK_LAYER}")
print(f"Probe (train/test) peak: {peak_tt['test_acc']:.3f} @ layer {PEAK_LAYER_TT}")
print(f"Metadata control: {meta_acc:.3f} +/- {meta_std:.3f}")
print(f"Permuted control: {best_perm['mean']:.3f}")

print(f"\nGap (probe - metadata): {best_probe['mean'] - meta_acc:+.3f}")
print(f"Gap (probe - permuted): {best_probe['mean'] - best_perm['mean']:+.3f}")
print(f"Bootstrap CI for gap: [{ci_lower:.3f}, {ci_upper:.3f}]")
print(f"Bootstrap p-value: {p_value:.4f}")
print(f"Permutation test p-value: {perm_p_value:.4f}")

print(f"\nProjection gap @ peak layer: Cohen's d = {gaps_by_layer[PEAK_LAYER]:.3f}")
print(f"Max projection gap: d = {max(gaps_by_layer):.3f} @ layer {np.argmax(gaps_by_layer)}")

print(f"\nPCA (layer {PEAK_LAYER}):")
print(f"  Variance in PC1+PC2: {sum(pca.explained_variance_ratio_[:2]):.1%}")
print(f"  |cos| probe dir with PC1: {cos_with_pcs[0]:.3f}")
print(f"  2D decision boundary acc: {acc_2d:.3f}")

print(f"\nSub-analysis ({SUB_CONSTRAINT}, n={len(df_sub)}):")
print(f"  Probe: {best_sub['mean']:.3f} | Meta: {meta_sub_acc:.3f} | Gap: {best_sub['mean'] - meta_sub_acc:+.3f}")

print(f"\nLeave-one-out generalization:")
for _, r in df_loco.iterrows():
    if not np.isnan(r['test_acc']):
        print(f"  {r['held_out']}: {r['test_acc']:.3f}")

print(f"\nPer-token transition:")
print(f"  Peak accuracy: {np.nanmax(token_accs_arr):.3f} @ token {np.nanargmax(token_accs_arr)}")
print(f"  System end: ~token {median_sys_end} | User end: ~token {median_user_end}")

if EXTRACT_COMPONENTS:
    print(f"\nComponent probing:")
    for comp_name, df_c in component_results.items():
        best_c = df_c.loc[df_c["mean"].idxmax()]
        print(f"  {comp_name}: {best_c['mean']:.3f} @ layer {int(best_c['layer'])}")

print("\n" + "=" * 60)

MODEL: meta-llama/Llama-3.2-1B-Instruct
BACKEND: transformerlens
TOKEN POSITION: last_prompt
SAMPLES: 211 (SCR: 6.6%)

Probe (CV) peak: 0.933 +/- 0.133 @ layer 9
Probe (train/test) peak: 0.667 @ layer 9
Metadata control: 0.631 +/- 0.124
Permuted control: 0.514

Gap (probe - metadata): +0.303
Gap (probe - permuted): +0.419
Bootstrap CI for gap: [0.141, 0.467]
Bootstrap p-value: 0.0000
Permutation test p-value: 0.0000

Projection gap @ peak layer: Cohen's d = 4.846
Max projection gap: d = 6.980 @ layer 15

PCA (layer 9):
  Variance in PC1+PC2: 39.0%
  |cos| probe dir with PC1: 0.038
  2D decision boundary acc: 0.500

Sub-analysis (format, n=49):
  Probe: 1.000 | Meta: 0.489 | Gap: +0.511

Leave-one-out generalization:
  format: 0.870
  language: 0.684
  starting_word: 0.500

Per-token transition:
  Peak accuracy: 0.609 @ token 39
  System end: ~token 51 | User end: ~token 76

